# 组合优化原理和细节

本篇说明目标、约束与求解后端之间的数学协议，适合扩展优化组件时查阅。日常使用见 [基本框架](基本框架.ipynb)、[均值方差模型](均值方差模型.ipynb) 和 [风险预算模型](风险预算模型.ipynb)。

运行身份、请求截面与跨轮缓存遵循 [Core 原理和细节](../Core/原理和细节.ipynb)。

## 架构设计

组合优化的整体流程如下：

```
用户输入
  ├── mask (股票池)
  ├── expected_return (预期收益)
  ├── cov / factor_cov + factor_data + specific_risk (风险模型)
  ├── p0 (初始组合)
  └── bmk (基准组合)
        │
        ▼
  OptimizationObjective.genObjective()  →  {'type': 'Quadratic', 'Sigma': ..., 'Mu': ...}
        │
        ▼
  Constraint.genConstraint() × N       →  [{'type': 'Box', ...}, {'type': 'LinearEq', ...}, ...]
        │
        ▼
  PortfolioConstructor.solve()         →  (array(n) 权重, dict 求解信息)
        │
        ▼
  CVXPC._solve()                       →  cvxpy 建模 + 求解器求解
```

**核心设计模式**：
- **目标-约束分离**：优化目标与约束条件是独立的对象，可自由组合
- **数学形式标准化**：`genObjective()` 和 `genConstraint()` 产出标准化的数学描述（dict），解耦了业务逻辑和求解器
- **约束松弛机制**：约束条件支持 `DropPriority` 优先级，当求解失败时按优先级自动松弛约束

## 优化目标

### OptimizationObjective — 优化目标基类

所有优化目标都继承自 `OptimizationObjective`，需要实现 `genObjective()` 方法。该方法返回一个字典，描述优化目标的数学形式：

| 类型 | 返回格式 | 说明 |
|------|----------|------|
| `Linear` | `{'f': array(n,), 'type': 'Linear'}` | 线性目标：$\mathbf{f}^T\mathbf{x}$ |
| `Quadratic` | `{'Sigma': array(n,n), 'Mu': array(n,), 'type': 'Quadratic'}` | 二次目标：$\mathbf{x}^T\Sigma\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x}$ |
| `Quadratic`（因子模型） | `{'X': array(n,k), 'F': array(k,k), 'Delta': array(n,), 'Mu': array(n,), 'type': 'Quadratic'}` | 二次目标（因子形式）：$\mathbf{x}^T(XFX^T+\Delta)\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x}$ |
| `L1_Linear` | `Linear` + L1 惩罚项 | 含交易成本惩罚的线性目标 |
| `L1_Quadratic` | `Quadratic` + L1 惩罚项 | 含交易成本惩罚的二次目标 |
| `Risk_Budget` | `{'Sigma': ..., 'b': array(n,), 'type': 'Risk_Budget'}` | 风险预算目标 |
| `Max_Diversification` | `{'Sigma': ..., 'type': 'Max_Diversification'}` | 最大分散化目标 |
| `Sharpe` | `{'f': ..., 'Sigma': ..., 'type': 'Sharpe'}` | 最大夏普率目标 |

每个目标都包含 `minmax` 字段（`'min'` 或 `'max'`），表示最小化还是最大化。

## 约束条件

大多数组合构建最终都归结为满足一定约束条件的最优化问题。不同的组合构建问题的优化目标各不一样，但常见的约束条件基本属于以下范畴。

所有约束条件都继承自 `Constraint`，需要实现 `genConstraint()` 方法，返回标准化约束列表。每个约束都有 `DropPriority` 参数（默认 -1，表示不可松弛），当求解失败时按非负优先级数值从小到大依次松弛，同值约束一起释放。

### 约束条件的数学形式

| 类型 | 返回格式 | 对应数学形式 |
|------|----------|-------------|
| `Box` | `{'lb': array(n,), 'ub': array(n,), 'type': 'Box'}` | $\mathbf{lb} \le \mathbf{x} \le \mathbf{ub}$ |
| `LinearIn` | `{'A': array(m,n), 'b': array(m,), 'type': 'LinearIn'}` | $A\mathbf{x} \le \mathbf{b}$ |
| `LinearEq` | `{'Aeq': array(m,n), 'beq': array(m,), 'type': 'LinearEq'}` | $A_{eq}\mathbf{x} = \mathbf{b}_{eq}$ |
| `Quadratic` | `{'Sigma': array(n,n), 'Mu': array(n,), 'q': float, 'type': 'Quadratic'}` | $\mathbf{x}^T\Sigma\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x} \le q$ |
| `L1` | `{'c': array(n,), 'l': float, 'type': 'L1'}` | $\sum \|x_i - c_i\| \le l$ |
| `Pos` | `{'c_pos': array(n,), 'l_pos': float, 'type': 'Pos'}` | $\sum(x_i - c_i)^+ \le l_{pos}$ |
| `Neg` | `{'c_neg': array(n,), 'l_neg': float, 'type': 'Neg'}` | $\sum(x_i - c_i)^- \le l_{neg}$ |
| `NonZeroNum` | `{'b': array(n,), 'N': float, 'type': 'NonZeroNum'}` | $\sum\mathbb{I}(x_i \neq 0) \le N$ |